# Get to Know a Dataset: ALFRED (The Allele Frequency Database)

This notebook serves as a guided tour of the [ALFRED (The Allele Frequency Database)](https://registry.opendata.aws/alfred-allele-frequency-database) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

ALFRED was conceived and developed by Prof. Kenneth K. Kidd and Dr. Judith R. Kidd at Yale University's Department of Genetics. It provides allele frequency data on human population samples for scientific research and educational use, containing over 54 million allele frequency tables for 664,437 polymorphisms across more than 750 anthropologically defined populations.

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

At the top level of our S3 bucket, we have organized the ALFRED dataset with the following structure:

 1. **Dataset documentation** (README.md, License.txt)
 2. **Population data** organized by geographic regions:
    - `populations/` - Contains population metadata and sample information
    - `geographic_regions/` - Population samples organized by geographic location (Africa, Europe, Asia, etc.)
 3. **Polymorphism data** organized by chromosome:
    - `chromosomes/chr1/` through `chromosomes/chr22/` and `chromosomes/chrX/`
    - Each chromosome directory contains parquet files with allele frequency tables
 4. **Locus and site information**:
    - `loci/` - Gene/locus metadata and mapping information
    - `sites/` - SNP/polymorphism site details including dbSNP rs numbers
 5. **Metadata tables**:
    - `contributors/` - Information about data contributors
    - `typing_methods/` - Details about genotyping methodologies used
 
The dataset follows the ALFRED database schema with interconnected tables linking populations, samples, loci, sites, and allele frequencies. Full documentation for this dataset can be found at: https://alfred.med.yale.edu/

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# pandas >= 2.0.0
# pyarrow >= 10.0.0  # for parquet file support
# matplotlib >= 3.10.3 
# seaborn >= 0.12.0
# numpy >= 1.24.0

First we will import the Python libraries required throughout this notebook.

In [ ]:
# Import the libraries required for this notebook
# Built-ins
import json
from pprint import pprint
import io

# Installed libraries
import boto3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from botocore import UNSIGNED
from botocore.config import Config

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see the main organizational structure of the ALFRED dataset.

In [ ]:
# Location of the S3 bucket for this dataset (placeholder - actual bucket TBD)
bucket = "alfred-allele-frequency-database"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
try:
    for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
        print(item['Prefix'])
except Exception as e:
    print(f"Note: S3 bucket not yet created. Expected structure:")
    print("chromosomes/")
    print("populations/")
    print("geographic_regions/")
    print("loci/")
    print("sites/")
    print("contributors/")
    print("typing_methods/")

Looking into the chromosomes directory, we can see how the genetic data is organized by chromosome. Each chromosome contains parquet files with allele frequency data for all polymorphisms on that chromosome.

In [ ]:
# List the chromosome directories
try:
    for item in s3.list_objects_v2(Bucket=bucket, Prefix='chromosomes/', Delimiter='/', MaxKeys=25)['CommonPrefixes']:
        print(item['Prefix'])
except Exception as e:
    print("Expected chromosome structure:")
    for i in range(1, 23):
        print(f"chromosomes/chr{i}/")
    print("chromosomes/chrX/")
    print("Note: ALFRED does not contain Y chromosome data")

Let's examine the structure within a specific chromosome directory to understand how the allele frequency data files are organized.

In [ ]:
# List the files within a chromosome directory (using chr1 as example)
try:
    for item in s3.list_objects_v2(Bucket=bucket, Prefix='chromosomes/chr1/', MaxKeys=10)['Contents']:
        print(item['Key'])
except Exception as e:
    print("Expected file structure within chromosomes/chr1/:")
    print("chromosomes/chr1/allele_frequencies_chr1_part_001.parquet")
    print("chromosomes/chr1/allele_frequencies_chr1_part_002.parquet")
    print("chromosomes/chr1/sites_chr1.parquet")
    print("chromosomes/chr1/loci_chr1.parquet")
    print("...")

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Our dataset uses **Parquet** format for storing allele frequency tables and associated metadata. Parquet is a columnar storage format that offers several advantages for genomic data:

**Why Parquet for ALFRED:**
- **Efficient compression**: Reduces storage costs for large-scale genomic data
- **Fast querying**: Columnar format enables quick filtering and aggregation
- **Schema evolution**: Supports adding new fields as the database grows
- **Cross-platform compatibility**: Works with Python, R, Java, and many other languages
- **Metadata preservation**: Maintains data types and column information

**Data stored in Parquet format:**
- **Allele frequency tables**: Population name, sample information, polymorphism details, and frequency values
- **Population metadata**: Geographic regions, sample sizes, contributor information
- **Polymorphism data**: dbSNP rs numbers, chromosomal positions, allele symbols
- **Locus information**: Gene symbols, chromosomal locations, functional annotations

**Working with Parquet files:**
- **Python**: Use `pandas.read_parquet()` or `pyarrow.parquet.read_table()`
- **R**: Use the `arrow` package with `read_parquet()`
- **AWS Services**: Amazon Athena, Amazon Redshift Spectrum, and AWS Glue natively support Parquet
- **Big Data Tools**: Apache Spark, Dask, and other distributed computing frameworks work seamlessly with Parquet

The tabular structure makes it easy to filter by population, chromosome, or specific polymorphisms, enabling efficient analysis of population genetics data.

### Q: Can you show us an example of downloading and loading data from your dataset?

As an example, let's load and examine allele frequency data for the SNP rs4653002 in the Yoruba population, which corresponds to the example data provided in our README.

In [ ]:
# Since the S3 bucket is not yet created, we'll create a sample dataset
# that matches the expected ALFRED data structure

# Create sample allele frequency data based on the README example
sample_data = {
    'Population_Name': ['Yoruba', 'Yoruba'],
    'PopulationUID': ['PO000036J', 'PO000036J'],
    'Sample_Name': ['Yoruba, HGDP-CEPH', 'Yoruba, HGDP-CEPH'],
    'SampleUID': ['SA001468T', 'SA001468T'],
    'NumberOfChrom': [50, 50],
    'Locus_Symbol': ['A3GALT2P', 'A3GALT2P'],
    'LocusUID': ['LO362836C', 'LO362836C'],
    'Site_Name': ['rs4653002', 'rs4653002'],
    'SiteUID': ['SI424900T', 'SI424900T'],
    'Typed_Sample': [48, 48],
    'Allele_Symbol': ['C', 'T'],
    'Allele_Frequency': [0.85, 0.15]
}

# Convert to DataFrame (simulating loading from parquet)
df_sample = pd.DataFrame(sample_data)

print("Sample ALFRED allele frequency data:")
print(df_sample)

Let's examine the structure of our allele frequency data. Each row represents one allele at a specific polymorphic site in a particular population sample.

In [ ]:
# Examine the data structure and types
print("Data shape:", df_sample.shape)
print("\nColumn information:")
print(df_sample.info())
print("\nSample statistics:")
print(df_sample.describe())

Now let's create a more comprehensive sample dataset to demonstrate the kind of analysis possible with ALFRED data. We'll simulate allele frequency data for multiple populations and SNPs.

In [ ]:
# Create a larger sample dataset for demonstration
np.random.seed(42)  # For reproducible results

# Define populations from different geographic regions
populations = [
    ('Yoruba', 'PO000036J', 'Africa'),
    ('Han Chinese', 'PO000045K', 'East Asia'),
    ('European American', 'PO000012C', 'Europe'),
    ('Maya', 'PO000078M', 'Northern America'),
    ('Papuan', 'PO000089P', 'S. Pacific/Oceania')
]

# Define some example SNPs
snps = [
    ('rs4653002', 'A3GALT2P', 'LO362836C'),
    ('rs1234567', 'APOE', 'LO123456A'),
    ('rs7890123', 'BRCA1', 'LO789012B'),
    ('rs4567890', 'TP53', 'LO456789T')
]

# Generate sample data
sample_records = []
for pop_name, pop_uid, region in populations:
    for snp_id, locus, locus_uid in snps:
        # Generate random allele frequencies that sum to 1
        freq1 = np.random.beta(2, 2)  # Beta distribution for realistic allele frequencies
        freq2 = 1 - freq1
        
        sample_size = np.random.randint(40, 100)  # Random sample size
        
        # Add records for both alleles
        for allele, freq in [('A', freq1), ('G', freq2)]:
            sample_records.append({
                'Population_Name': pop_name,
                'PopulationUID': pop_uid,
                'Geographic_Region': region,
                'Sample_Name': f'{pop_name}, HGDP-CEPH',
                'SampleUID': f'SA{np.random.randint(100000, 999999)}T',
                'NumberOfChrom': sample_size,
                'Locus_Symbol': locus,
                'LocusUID': locus_uid,
                'Site_Name': snp_id,
                'SiteUID': f'SI{np.random.randint(100000, 999999)}T',
                'Typed_Sample': sample_size - np.random.randint(0, 5),
                'Allele_Symbol': allele,
                'Allele_Frequency': round(freq, 3)
            })

# Create comprehensive DataFrame
df_comprehensive = pd.DataFrame(sample_records)

print(f"Generated sample dataset with {len(df_comprehensive)} records")
print(f"Covering {len(populations)} populations and {len(snps)} SNPs")
print("\nFirst few records:")
print(df_comprehensive.head(10))

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

Let's create visualizations that showcase the power of ALFRED data for understanding human genetic diversity. We'll start by examining allele frequency differences across populations.

In [ ]:
# Create a visualization showing allele frequency variation across populations
# Focus on 'A' allele frequencies for each SNP across populations

# Filter for 'A' alleles only
df_a_alleles = df_comprehensive[df_comprehensive['Allele_Symbol'] == 'A'].copy()

# Create a pivot table for easier plotting
freq_matrix = df_a_alleles.pivot_table(
    index='Site_Name', 
    columns='Population_Name', 
    values='Allele_Frequency',
    aggfunc='first'
)

# Create the heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(freq_matrix, 
            annot=True, 
            cmap='RdYlBu_r', 
            center=0.5,
            fmt='.3f',
            cbar_kws={'label': 'Allele Frequency'})

plt.title('Allele Frequency Variation Across Human Populations\n(A allele frequencies for sample SNPs)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Population', fontsize=12)
plt.ylabel('SNP (dbSNP ID)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("This heatmap shows how allele frequencies vary across different human populations.")
print("Darker red indicates higher frequency, darker blue indicates lower frequency.")
print("Such patterns reflect human migration history and population structure.")

Now let's create a visualization showing the geographic distribution of genetic diversity, which is a key application of ALFRED data in understanding human evolution.

In [ ]:
# Calculate heterozygosity (genetic diversity) for each population
# Heterozygosity = 2 * p * (1-p) where p is allele frequency

def calculate_heterozygosity(group):
    """Calculate expected heterozygosity for a group of alleles at one site"""
    frequencies = group['Allele_Frequency'].values
    # Expected heterozygosity = 1 - sum(p_i^2)
    return 1 - sum(freq**2 for freq in frequencies)

# Calculate heterozygosity for each population-SNP combination
heterozygosity_data = []
for (pop, snp), group in df_comprehensive.groupby(['Population_Name', 'Site_Name']):
    het = calculate_heterozygosity(group)
    region = group['Geographic_Region'].iloc[0]
    heterozygosity_data.append({
        'Population': pop,
        'SNP': snp,
        'Geographic_Region': region,
        'Heterozygosity': het
    })

df_het = pd.DataFrame(heterozygosity_data)

# Calculate average heterozygosity per population
avg_het = df_het.groupby(['Population', 'Geographic_Region'])['Heterozygosity'].mean().reset_index()

# Create bar plot
plt.figure(figsize=(12, 8))
bars = plt.bar(avg_het['Population'], avg_het['Heterozygosity'], 
               color=sns.color_palette("husl", len(avg_het)))

# Add region labels
for i, (bar, region) in enumerate(zip(bars, avg_het['Geographic_Region'])):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             region, ha='center', va='bottom', rotation=45, fontsize=9)

plt.title('Genetic Diversity (Heterozygosity) Across Human Populations', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Population', fontsize=12)
plt.ylabel('Average Expected Heterozygosity', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.ylim(0, max(avg_het['Heterozygosity']) * 1.2)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print("Expected heterozygosity is a measure of genetic diversity within populations.")
print("Higher values indicate more genetic variation.")
print("This pattern often reflects the 'Out of Africa' migration model,")
print("where African populations typically show the highest diversity.")

Let's also create a visualization showing the distribution of sample sizes across populations, which is important for understanding the statistical power of different population samples in ALFRED.

In [ ]:
# Analyze sample sizes across populations
sample_sizes = df_comprehensive.groupby('Population_Name')['Typed_Sample'].first().reset_index()
sample_sizes = sample_sizes.sort_values('Typed_Sample', ascending=True)

# Create horizontal bar plot
plt.figure(figsize=(10, 6))
bars = plt.barh(sample_sizes['Population_Name'], sample_sizes['Typed_Sample'],
                color=sns.color_palette("viridis", len(sample_sizes)))

# Add value labels on bars
for bar in bars:
    width = bar.get_width()
    plt.text(width + 1, bar.get_y() + bar.get_height()/2, 
             f'{int(width)}', ha='left', va='center')

plt.title('Sample Sizes Across ALFRED Populations', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Number of Typed Samples', fontsize=12)
plt.ylabel('Population', fontsize=12)
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Sample sizes range from {sample_sizes['Typed_Sample'].min()} to {sample_sizes['Typed_Sample'].max()} individuals.")
print("Larger sample sizes provide more reliable allele frequency estimates.")
print("ALFRED contains samples from over 750 populations worldwide.")

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Research Question**: *How does genetic diversity decrease along human migration routes, supporting the "Out of Africa" model?*

One of the key applications of ALFRED data is testing evolutionary hypotheses about human migration. The "Out of Africa" model predicts that genetic diversity should be highest in African populations and decrease along migration routes due to serial founder effects. Let's demonstrate this analysis using our sample data.

**Methodology**:
1. Calculate expected heterozygosity for each population
2. Organize populations by their approximate distance from Africa
3. Test for correlation between geographic distance and genetic diversity
4. Visualize the pattern

In [ ]:
# Define approximate migration distances from Africa (in arbitrary units)
# Based on major human migration routes
migration_distances = {
    'Yoruba': 0,  # African origin
    'European American': 1,  # Early migration to Europe
    'Han Chinese': 2,  # Migration to East Asia
    'Papuan': 3,  # Migration to Oceania
    'Maya': 4  # Migration to Americas (via Beringia)
}

# Add migration distance to our heterozygosity data
avg_het['Migration_Distance'] = avg_het['Population'].map(migration_distances)

# Create scatter plot showing relationship
plt.figure(figsize=(10, 6))
scatter = plt.scatter(avg_het['Migration_Distance'], avg_het['Heterozygosity'], 
                     s=100, alpha=0.7, c=range(len(avg_het)), cmap='viridis')

# Add population labels
for i, row in avg_het.iterrows():
    plt.annotate(row['Population'], 
                (row['Migration_Distance'], row['Heterozygosity']),
                xytext=(5, 5), textcoords='offset points', fontsize=10)

# Fit and plot trend line
z = np.polyfit(avg_het['Migration_Distance'], avg_het['Heterozygosity'], 1)
p = np.poly1d(z)
plt.plot(avg_het['Migration_Distance'], p(avg_het['Migration_Distance']), 
         "r--", alpha=0.8, linewidth=2, label=f'Trend line (slope={z[0]:.3f})')

# Calculate correlation
correlation = np.corrcoef(avg_het['Migration_Distance'], avg_het['Heterozygosity'])[0,1]

plt.title('Genetic Diversity vs. Migration Distance from Africa\n(Supporting the "Out of Africa" Model)', 
          fontsize=14, fontweight='bold', pad=20)
plt.xlabel('Approximate Migration Distance from Africa', fontsize=12)
plt.ylabel('Expected Heterozygosity', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Correlation coefficient: {correlation:.3f}")
print(f"Trend line slope: {z[0]:.3f}")
print("\nInterpretation:")
if correlation < -0.5:
    print("Strong negative correlation supports the Out of Africa model.")
    print("Genetic diversity decreases with distance from Africa.")
elif correlation < 0:
    print("Moderate negative correlation suggests some support for the Out of Africa model.")
else:
    print("No clear pattern observed (note: this is simulated data).")
    
print("\nThis type of analysis using ALFRED data has been crucial in:")
print("- Confirming human migration patterns")
print("- Understanding population bottlenecks")
print("- Studying the effects of genetic drift")

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Research Challenge**: *Can we develop improved methods for inferring individual ancestry and admixture proportions using ALFRED's comprehensive population reference panels?*

**The Question**: While current ancestry inference methods work well for major continental groups, they often struggle with:
- Fine-scale population structure within continents
- Populations with complex admixture histories
- Understudied populations that are well-represented in ALFRED

**Why ALFRED is uniquely suited for this**:
- Over 750 anthropologically defined populations
- Extensive coverage of understudied populations
- Consistent genotyping across populations
- Geographic metadata for populations

**Recommendations for researchers**:

1. **Start with data exploration**:
   - Use PCA and ADMIXTURE analysis on ALFRED populations
   - Identify populations with unique allele frequency profiles
   - Focus on regions with high population diversity (e.g., Africa, Oceania)

2. **Methodological approaches**:
   - Implement machine learning methods (Random Forest, Neural Networks)
   - Develop hierarchical models that account for population structure
   - Use Bayesian approaches for uncertainty quantification

3. **Technical considerations**:
   - Handle missing data appropriately
   - Account for linkage disequilibrium between markers
   - Validate using cross-validation within ALFRED populations

4. **Computational resources**:
   - Use AWS services like SageMaker for machine learning
   - Leverage Athena for large-scale data querying
   - Consider distributed computing for population-scale analyses

5. **Validation strategy**:
   - Test on populations with known admixture (e.g., African Americans)
   - Compare with existing methods (STRUCTURE, ADMIXTURE)
   - Validate using independent datasets when possible

**Impact**: Success in this area could improve:
- Personalized medicine applications
- Forensic identification methods
- Understanding of human population history
- Representation of diverse populations in genomic research

**Getting started**: Begin by downloading allele frequency data for 50-100 well-characterized populations, implement a baseline method, and gradually expand to include more populations and sophisticated algorithms.

## Summary

This notebook has provided an introduction to the ALFRED (Allele Frequency Database) dataset, demonstrating:

- **Data organization**: How genetic data is structured by chromosomes, populations, and geographic regions
- **Data formats**: The use of Parquet files for efficient storage and analysis of genomic data
- **Analysis examples**: Visualization of genetic diversity patterns across human populations
- **Research applications**: Testing evolutionary hypotheses like the "Out of Africa" model
- **Future directions**: Opportunities for advancing ancestry inference methods

ALFRED's comprehensive coverage of human genetic diversity makes it an invaluable resource for population genetics research, evolutionary studies, and educational applications. The dataset's organization and format facilitate both targeted analyses of specific populations or genes and large-scale comparative studies across human populations.

For more information and to access the full dataset, visit the [ALFRED website](https://alfred.med.yale.edu/) and the [Registry of Open Data on AWS](https://registry.opendata.aws/).

---
*Remember to clear all outputs before committing to your repository*